<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Exercises 8 to 12. Second opinions, and the same loop on a different molecule</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Modules 5 to 8. Five tasks, 63 minutes, ending with a paper you have not read.</em>

Everything so far has verified a generated object against a fixed rule: is this bond
length plausible, is this ring flat, how far apart are these two atoms. Rules are
cheap and they catch a lot. What they cannot tell you is whether a pose is right,
because there is no rule for that; there is only a second method that never saw the
first method's answer, and the size of the disagreement between them.

Five tasks. X8 asks what a retention rate is a rate of, when two defensible
measurements of the same thing disagree by a factor of five. X9 puts a docking model
in as the second opinion and asks what the two methods agree and disagree about. X10
hands you a model whose output has two heads and invites you to read the wrong one.
X11 does the whole loop again on a protein instead of a small molecule, and asks you
to say which model plays which role. X12 sends you to a paper.

X10 is the second of the two failure-discovery exercises, and like X5 it works once.
Do it before reading the solution.

<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Objectives</strong>
<ol style="margin:8px 0 0;padding-left:22px;font-size:15.5px;color:#241c1d;">
<li style="margin:6px 0;">Measure a retention rate two ways and explain why the two disagree</li>
<li style="margin:6px 0;">Use an independent second method as a check and separate what it confirms from what it disputes</li>
<li style="margin:6px 0;">Read a two-headed model output in the right order and refuse the number that is not defined</li>
<li style="margin:6px 0;">Map a verification loop from one molecular domain onto another</li>
</ol>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #6B6461;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#6B6461;">Runtime</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Everything in this notebook runs on a laptop CPU in well under a minute of compute. There is no download, no GPU, no API key and no network call; every number you produce comes from a file shipped inside this bundle.</span>
</div>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #B4690E;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#B4690E;">Caution</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Read every metrics CSV in this bundle with <code>gv.read_metrics_csv</code>. The files carry provenance as leading <code>#</code> lines, and <code>#</code> is also the SMILES symbol for a triple bond, so passing <code>comment='#'</code> to pandas silently truncates any row holding a nitrile: 16 cells go NaN in <code>boltz2_comparison.csv</code> and 26 in <code>novelty_diffsbdd.csv</code>, with no error raised.</span>
</div>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Each task ends with a self-check cell. It tells you whether your answer sits inside the tolerance for that task and, when it does not, which way to look. It never prints the answer. The worked solutions live in <code>solutions/</code>; open them after you have an answer of your own, not before.</span>
</div>

In [ ]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

In [ ]:
# Shared setup.
import json
import numpy as np
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X8&nbsp;&middot;&nbsp;intermediate&nbsp;&middot;&nbsp;10 min&nbsp;&middot;&nbsp;M5&nbsp;&middot;&nbsp;O14, O15, O16</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Did the scaffold survive</h2>

Inpainting is generation with part of the answer already filled in. The run behind
this task fixed the nine-atom purine core of caffeine, held those atoms at their
crystallographic coordinates, and let the model grow new chemistry around them.
Ten molecules came back.

Report the retention rate two ways.

1. By coordinate: did the fixed atoms stay where they were pinned?
2. By pattern: does a strict aromatic-purine SMARTS still match the resulting molecule?

Then explain the difference, which is the point of the task. Finally, using the
scheme of what is held fixed and where the reverse chain starts, describe a fourth
task you could ask this same frozen checkpoint to do, and say what you would hold
fixed.

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">The recorded QED-optimization cell has a trailing print that reads a variable assigned only on the fresh-compute branch, so on the cached path it raises NameError after the figure renders. The figure and every plotted value are unaffected. We read <code>best_qed</code> from <code>diffsbdd_optimize_metrics.json</code> instead of reusing that print.</span>
</div>

In [ ]:
# X8, part 1. Retention by coordinate.
inpaint = json.loads((DATA / "diffsbdd_inpaint_metrics.json").read_text())
print("mode:", inpaint["mode"])
print("fixed scaffold:", inpaint["fixed_scaffold_smiles"])
print("requested:", inpaint["n_requested"], "| valid:", inpaint["n_valid"])

records = inpaint["records"]
median_coord_retention = None      # YOUR CODE HERE (median of scaffold_coord_retention)

# X8, part 2. Retention by SMARTS pattern.
#
# The strict aromatic purine pattern. A match means RDKit perceived the resulting
# molecule as still containing an aromatic purine after bond orders were assigned.
PURINE = Chem.MolFromSmarts("c1ncc2ncnc2n1")

inpainted = gv.read_sdf(DATA / "diffsbdd_inpainted.sdf")
n_smarts_keep = None               # YOUR CODE HERE (how many of the ten match?)
smarts_retention_pct = None        # YOUR CODE HERE (as a percentage)

print("\nby coordinate:", median_coord_retention)
print("by SMARTS:", n_smarts_keep, "of", len(inpainted),
      "=", smarts_retention_pct, "percent")

In [ ]:
# X8, part 3. Explain the difference. Inpainting promises something specific about
# the fixed atoms. What does it promise, and what does it not?
retention_explanation = ""     # YOUR ANSWER HERE

# X8, part 4. A fourth task for the same frozen checkpoint.
#
# The three tasks this checkpoint already performed:
#
#   task                     held fixed                    chain starts from
#   ---------------------------------------------------------------------------
#   de novo design           the pocket                    pure noise
#   QED optimization         the pocket                    a noised seed ligand
#   scaffold inpainting      pocket + 9 purine atoms       pure noise, masked
#   YOUR FOURTH TASK         ?                             ?
#
fourth_task = ""               # YOUR ANSWER HERE: name what is held fixed and
                               # where the chain starts.

In [ ]:
# X8. The optimization number, read from the metrics file rather than from the
# published cell's trailing print.
opt = json.loads((DATA / "diffsbdd_optimize_metrics.json").read_text())
best_qed = opt["best_qed"]
print(f"best QED after {opt['evolution_steps']} generations: {best_qed:.4f} "
      f"(seed ligand caffeine at {opt['seed_qed']:.4f})")

gv.check_value("X8.coord_retention", median_coord_retention)
gv.check_value("X8.smarts_keep", n_smarts_keep)
gv.check_value("X8.smarts_pct", smarts_retention_pct)
gv.check_value("X8.best_qed", best_qed)
gv.check_written("X8.explanation", retention_explanation,
                  "Inpainting fixes positions and elements. What assigns the bonds, "
                  "and when?")
gv.check_written("X8.fourth_task", fourth_task,
                  "Name the mask and the chain start, rather than only the goal.")
gv.checklist_summary("X8")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X9&nbsp;&middot;&nbsp;intermediate&nbsp;&middot;&nbsp;10 min&nbsp;&middot;&nbsp;M6&nbsp;&middot;&nbsp;O17, O18</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Ask a second method</h2>

DiffSBDD produced twenty ligands and a pose for each. DiffDock, a different model
trained on a different task by a different group, was then asked to dock each of
those twenty molecules into the same receptor from scratch, with no knowledge of
where DiffSBDD had put it. That gives two independent opinions about where each
molecule sits.

Answer three questions and then draw one conclusion.

1. How many of the twenty top-ranked re-docked poses fall within 2 angstroms of the
   pose the generator produced?
2. How many re-docked centroids sit within 5 angstroms of the co-crystallized caffeine?
3. On the identical PoseBusters battery, which checks does each pose source fail?

Then write the one sentence those three answers support together, and say what it
implies about how much to trust a single rendered pose.

Avoid reading that disagreement as proof that one method failed. No experimental
pose exists for an unsynthesized molecule, so neither answer is ground truth; you
have two estimates and the distance between them.

In [ ]:
# X9. The re-docking table alongside the generated-pose battery.
dd = gv.read_metrics_csv(DATA / "diffdock_redock_posebusters.csv")
pb = gv.read_metrics_csv(DATA / "posebusters_diffsbdd.csv")
print(dd[["mol_id", "diffdock_top_confidence", "rmsd_generated_vs_redocked_A",
          "min_rmsd_any_pose_A", "redock_centroid_dist_to_caffeine_A",
          "pb_valid"]].to_string(index=False))

In [ ]:
# X9, question 1. Agreement on the pose.
n_within_2_top = None        # YOUR CODE HERE, using rmsd_generated_vs_redocked_A
n_within_2_any = None        # YOUR CODE HERE, using min_rmsd_any_pose_A
median_rmsd = None           # YOUR CODE HERE

# X9, question 2. Agreement on the site.
n_centroid_within_5 = None   # YOUR CODE HERE, using redock_centroid_dist_to_caffeine_A
median_centroid = None       # YOUR CODE HERE

print("within 2 A on the top pose:", n_within_2_top, "of", len(dd))
print("within 2 A allowing any returned pose:", n_within_2_any, "of", len(dd))
print("median generated-vs-redocked RMSD:", median_rmsd)
print("re-docked centroids within 5 A of caffeine:", n_centroid_within_5, "of", len(dd))
print("median re-docked centroid distance:", median_centroid)

In [ ]:
# X9, question 3. The same battery, the two pose sources side by side.
CHECKS = ["bond_lengths", "bond_angles", "minimum_distance_to_protein",
          "internal_steric_clash", "volume_overlap_with_protein"]

print(f"{'check':<34s} {'generated':>12s} {'re-docked':>12s}   (percent passing)")
for c in CHECKS:
    print(f"{c:<34s} {pb[c].mean() * 100:>11.0f}% {dd[c].mean() * 100:>11.0f}%")

n_valid_generated = None     # YOUR CODE HERE (fully valid, generated poses)
n_valid_redocked = None      # YOUR CODE HERE (fully valid, re-docked poses)
mindist_fail_pct = None      # YOUR CODE HERE: percent of re-docked poses FAILING
                             # minimum_distance_to_protein

print()
print("fully valid:", n_valid_generated, "generated |", n_valid_redocked, "re-docked")

# DiffDock's own top-pose confidence.
print("DiffDock median top-pose confidence:",
      round(float(dd.diffdock_top_confidence.median()), 4),
      "| negative for", int((dd.diffdock_top_confidence < 0).sum()), "of", len(dd))

In [ ]:
# X9, the conclusion. One sentence covering all three answers, plus what it means
# for a single rendered pose in a figure.
second_opinion_conclusion = ""    # YOUR ANSWER HERE

gv.check_value("X9.within2_top", n_within_2_top)
gv.check_value("X9.within2_any", n_within_2_any)
gv.check_value("X9.median_rmsd", median_rmsd)
gv.check_value("X9.cent_within5", n_centroid_within_5)
gv.check_value("X9.cent_median", median_centroid)
gv.check_value("X9.pb_generated", n_valid_generated)
gv.check_value("X9.pb_redocked", n_valid_redocked)
gv.check_value("X9.mindist_fail_pct", mindist_fail_pct)
gv.check_written("X9.conclusion", second_opinion_conclusion,
                  "The two methods agree strongly about one thing and disagree "
                  "about another. Does your sentence separate them?")
gv.checklist_summary("X9")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X10&nbsp;&middot;&nbsp;advanced&nbsp;&middot;&nbsp;18 min&nbsp;&middot;&nbsp;M7&nbsp;&middot;&nbsp;O19, O20, O21</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Read the wrong number on purpose</h2>

Boltz-2 is a co-folding model: give it a protein sequence and a ligand and it
predicts the structure of the complex, and alongside that it predicts affinity. The
affinity prediction has two heads. One is a binary binder-versus-decoy probability.
The other is a continuous value on a log10 IC50 in micromolar scale.

The shipped table has both, for seven molecules, under two different implementations
of the same model.

Four parts, in this order. Do not skip ahead.

1. Convert the continuous value to micromolar for caffeine and for the FlowMol3
   molecule. Report both.
2. Report each molecule's binder probability.
3. Something is wrong with the conclusion part one invites. Say what it is.
4. Compute the rank correlation between the two implementations, identify which
   molecules change rank, and say what that does to the sentence "design 2 was our
   best candidate".

Part four is the sharper half. The two implementations agree on every summary
statistic you would put in a table. Look at what they disagree about anyway.

In [ ]:
# X10. The Boltz-2 comparison table. Seven molecules, two implementations.
bz = gv.read_metrics_csv(DATA / "boltz2_comparison.csv")
print(bz[["mol_id", "qed",
          "modal_affinity_probability_binary", "nim_affinity_probability_binary",
          "modal_affinity_pred_value_log10_IC50_uM",
          "modal_rank", "nim_rank"]].to_string(index=False))

In [ ]:
# X10, part 1. Convert the continuous affinity value to micromolar.
#
# The column is log10 of IC50 in micromolar, so the conversion is 10 ** value.
caffeine_row = bz[bz.mol_id == "caffeine_control"].iloc[0]
flowmol_row = bz[bz.mol_id == "flowmol3_ref"].iloc[0]

caffeine_uM = None      # YOUR CODE HERE
flowmol_uM = None       # YOUR CODE HERE

print(f"caffeine:            {caffeine_uM} micromolar")
print(f"FlowMol3 molecule:   {flowmol_uM} micromolar")

In [ ]:
# X10, part 2. Now the other head.
caffeine_prob = None    # YOUR CODE HERE
flowmol_prob = None     # YOUR CODE HERE

print("binder probability, all seven molecules:")
for _, r in bz.sort_values("modal_affinity_probability_binary",
                           ascending=False).iterrows():
    print(f"  {r.mol_id:<20s} {r.modal_affinity_probability_binary:.4f}")

THRESHOLD = 0.5
n_above_threshold = None     # YOUR CODE HERE: how many clear the 0.5 binary threshold?
print(f"\nabove the {THRESHOLD} threshold: {n_above_threshold} of {len(bz)}")

In [ ]:
# X10, part 3. Something is wrong with what part one invites you to conclude.
#
# Part one produced two micromolar figures. Taken at face value they say the
# generated molecule is roughly twice as potent as the drug that is known to bind
# this receptor. Part two produced the probabilities. Put them together.
#
# State the problem in two or three sentences, and say which of the two numbers you
# would report.
whats_wrong = ""        # YOUR ANSWER HERE

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #B4690E;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#B4690E;">Caution</strong><br>
<span style="color:#3d3436;font-size:14.5px;">The micromolar figure you just computed for the FlowMol3 molecule is INVALID and must never be quoted as a potency, here or anywhere. The continuous head is calibrated on molecules already known to be active; it is not defined for a molecule the binary head has placed below the threshold, and this one scores 0.09. It appears in this exercise only as the thing you compute and then discard, which is the entire point of the task. Read the probability first, every time.</span>
</div>

In [ ]:
# X10, part 4. The two implementations.
#
# Every molecule was scored twice: once by a pinned local build and once by a hosted
# service running the same model. The columns are prefixed modal_ and nim_.
from scipy.stats import spearmanr

spearman_rho = None       # YOUR CODE HERE: rank correlation of the two probability
                          # columns
mean_abs_delta = None     # YOUR CODE HERE: mean absolute probability difference
n_ranks_identical = None  # YOUR CODE HERE: how many of the seven keep their rank?

print("Spearman rho:", spearman_rho)
print("mean absolute probability difference:", mean_abs_delta)
print("identical ranks:", n_ranks_identical, "of", len(bz))

print()
print("where the two implementations disagree:")
moved = bz[bz.modal_rank != bz.nim_rank]
print(moved[["mol_id", "modal_affinity_probability_binary",
             "nim_affinity_probability_binary", "modal_rank",
             "nim_rank"]].to_string(index=False))

In [ ]:
# X10, part 4 continued.
#
# Which two designs swap? Give their indices as digits, lowest first, e.g. "13".
swap_pair = ""            # YOUR ANSWER HERE

# And what does that do to the sentence "design 2 was our best candidate"?
best_candidate_claim = ""  # YOUR ANSWER HERE

gv.check_value("X10.caffeine_uM", caffeine_uM)
gv.check_value("X10.flowmol_uM", flowmol_uM)
gv.check_value("X10.caffeine_prob", caffeine_prob)
gv.check_value("X10.flowmol_prob", flowmol_prob)
gv.check_value("X10.n_above_gate", n_above_threshold)
gv.check_value("X10.spearman", spearman_rho)
gv.check_value("X10.mean_abs_delta", mean_abs_delta)
gv.check_value("X10.ranks_identical", n_ranks_identical)
gv.check_text("X10.swap_pair", swap_pair)
gv.check_written("X10.whats_wrong", whats_wrong,
                  "Which head is a threshold and which is a magnitude, and what is the "
                  "magnitude conditional on?")
gv.check_written("X10.claim", best_candidate_claim,
                  "Is the claim about the molecules, about Boltz-2, or about "
                  "something narrower than either?")
gv.checklist_summary("X10")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X11&nbsp;&middot;&nbsp;intermediate&nbsp;&middot;&nbsp;10 min&nbsp;&middot;&nbsp;M8&nbsp;&middot;&nbsp;O22, O23</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">The same test, a different molecule</h2>

Everything so far has been a small molecule. This task changes the object and keeps
the loop. RFdiffusion generates a protein backbone, ProteinMPNN designs a sequence
that might fold into it, and ESMFold predicts what that sequence folds into.
The self-consistency RMSD is the distance between the backbone you asked for and the
structure you got. A design is called designable when that distance is under 2
angstroms and the fold prediction is confident, meaning a pLDDT above 70.

Three parts.

1. From the 24-design table, count how many designs are inside the designable box
   and report the median self-consistency RMSD.
2. Compute the designable fraction at each of the four backbone lengths, and say
   whether the pipeline degrades with length.
3. Write one sentence mapping this loop onto the re-docking cross-check from X9:
   which model plays which role.

Part three is the transfer check and the single most diagnostic item in the set. A
learner who cannot write it has learned two pipelines rather than one loop.

In [ ]:
# X11, part 1. The 24-design self-consistency table.
sc = json.loads((DATA / "selfconsistency_full.json").read_text())
designs = sc["all"]
print(f"{len(designs)} designs at 80 residues")
print("criteria: scRMSD < 2.0 angstroms AND pLDDT > 70")

n_designable = None       # YOUR CODE HERE
median_scrmsd = None      # YOUR CODE HERE

print("designable:", n_designable, "of", len(designs))
print("median scRMSD:", median_scrmsd)
print("worst scRMSD:", max(d["scrmsd"] for d in designs))

In [ ]:
# X11, part 2. Does it degrade with length?
scaling = json.loads((DATA / "designability_scaling.json").read_text())
per_design = scaling["per_design"]
print(scaling["description"])

designable_fraction = {}
for L in scaling["lengths"]:
    # YOUR CODE HERE: the percentage of designs at length L that are designable.
    designable_fraction[L] = None

for L, pct in designable_fraction.items():
    print(f"  {L:>3} residues   {pct} percent designable")

# One word: does the designable fraction rise or fall with length?
length_trend = ""         # YOUR ANSWER HERE

In [ ]:
# X11, part 3. The transfer sentence. Map this loop onto X9's re-docking check.
# Which model here plays the generator, and which plays the independent second
# method? What quantity here is the analogue of the pose RMSD?
transfer_sentence = ""    # YOUR ANSWER HERE

gv.check_value("X11.designable", n_designable)
gv.check_value("X11.median_scrmsd", median_scrmsd)
for L in [60, 90, 120, 150]:
    gv.check_value(f"X11.frac{L}", designable_fraction.get(L))
gv.check_text("X11.trend", length_trend)
gv.check_written("X11.transfer", transfer_sentence,
                  "Name three things: the generator, the second method, and the "
                  "disagreement quantity.")
gv.checklist_summary("X11")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X12&nbsp;&middot;&nbsp;advanced&nbsp;&middot;&nbsp;15 min&nbsp;&middot;&nbsp;M8&nbsp;&middot;&nbsp;O24</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Read a paper you have not seen</h2>

No compute. Fifteen minutes of reading and one filled table.

Pick one of the papers below and read only its abstract. Identify its generator, its
verifier, and the number the verifier reports. Then say which module of this
curriculum prepared you to read that number, and name one thing the abstract does not
tell you that you would now want to check.

This is the entry-point criterion made assessable. If the eight modules did their
job, an abstract you have never seen should now decompose into a generator, a
verifier and a number, and you should have at least one question about how the
verifier was set up.

**The four papers.** All four are frontier work from 2023 to 2026 that uses the
generate-then-verify loop this resource teaches.

1. **Benchmarking Generated Poses: How Rational is Structure-based Drug Design with
   Generative Models?** Harris et al., 2023. arXiv:2308.07413.
2. **PoseBusters: AI-based docking methods fail to generate physically valid poses or
   generalise to novel sequences.** Buttenschoen, Morris and Deane, 2024. Chemical
   Science, doi:10.1039/d3sc04185a.
3. **Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.** Passaro
   et al., 2025. bioRxiv, doi:10.1101/2025.06.14.659707.
4. **FLOWR: flow matching for structure-aware de novo, interaction- and
   fragment-based ligand generation.** Cremer et al., 2026. Nature Computational
   Science, doi:10.1038/s43588-026-00998-8.

A fifth, if you want a generator with no receptor in the loop: **TABASCO: A Fast,
Simplified Model for Molecular Generation with Improved Physical Quality.** Vonessen
et al., 2025. arXiv:2507.00899.

In [ ]:
# X12. Fill one row. Five fields, all strings.
paper_read = ""          # which paper did you pick?
generator = ""           # what generates the candidate?
verifier = ""            # what checks it?
reported_number = ""     # what quantity does the verifier report?
prepared_by_module = ""  # which module of this curriculum taught you to read it?
what_is_missing = ""     # one thing the abstract does not tell you that you would
                         # now want to check

for field, value in [("paper", paper_read), ("generator", generator),
                     ("verifier", verifier), ("reported number", reported_number),
                     ("module", prepared_by_module), ("missing", what_is_missing)]:
    mark = "ok" if str(value).strip() else "  "
    print(f"  [{mark}] {field:<16s} {value}")

**How this one is graded.** There is no key, because the answer depends on which
paper you picked. The rubric is five points: one each for correctly naming the
generator, the verifier and the reported metric, one for naming a module that
teaches that metric, and one for a missing-information question that is
answerable and specific. Three of five is a pass.

"How was the receptor prepared" is a specific question. "More detail would be
helpful" is not.

In [ ]:
# X12 self-check. Completeness only; the rubric is yours to apply.
answered = sum(1 for v in [paper_read, generator, verifier, reported_number,
                           prepared_by_module, what_is_missing] if str(v).strip())
print(f"{answered} of 6 fields filled")
gv.check_written("X12.missing", what_is_missing,
                  "Is your question answerable from the paper's methods section?")
gv.checklist_summary()

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">You have finished the set</h2>

Thirteen tasks. Two of them, X5 and X10, handed you a result that looked fine and let
you find out for yourself that it was not, and those two are the ones to remember.
The rest built the habits that made those two findable: predict before you measure,
name the confound before you explain the gap, state the threshold your ratio is
relative to, name the definition behind your number, and ask a second method when no
rule can settle the question.

Generation and verification form one loop: a generator proposes a structure, a
verifier tests where it breaks, and your conclusions live in the gap between
them. X11 checked that transfer across molecular classes, and X12 carries it into
the next paper you read.

Worked solutions for every task are in `solutions/`.